In [2]:
!pip install ddgs
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.2 MB/s eta 0:00:00


# EJEMPLO SIN CONTEXTO ACTUALIZADO

In [ ]:
 # Importamos la libreria de colab para poder usar el key_vault , si se ejecuta en local se puede optar por usar un .env
from google.colab import userdata
api_key = userdata.get('GROQ_API_KEY')
 # Importamos la libreria para comunicarnos con la api
from groq import Groq

#Creamos la conexion con la funcion Groq
client = Groq(api_key=api_key)


In [9]:
# Importamos la libreria para comunicarnos con la api
from groq import Groq

#Creamos la conexion con la funcion Groq
client = Groq(api_key=api_key)

# client es nuestro objeto conexion para comunicarnos con la API
# utilizamos el metodo chat.completions.create para la consulta a
response = client.chat.completions.create(

model="llama-3.1-8b-instant",

messages=[

{"role": "user", "content": "Quien fue el ganador, y por cuanto gano, del partido de futbol de Argentina contra Honduras en Junio del 26"}

]
)

print(response.choices[0].message.content)

Lo siento, pero no tengo información específica sobre un partido de fútbol entre Argentina y Honduras en Junio del 26 sin una fecha de año específica. Sin embargo, puedo sugerirte algunas opciones para encontrar la respuesta que buscas:

1. Verifica la fuente de noticias: Puedes buscar en sitios de noticias como ESPN, UEFA, FIFA, Clarín, La Nación, TyC Sports, etc. para ver si hay un artículo o reporte sobre un partido entre Argentina y Honduras en la fecha que mencionas.
2. Consulta el calendario de partidos: Puedes buscar en sitios web como ESPN Deportes, FIFA, UEFA, etc. para ver si hay un calendario de partidos que incluya la fecha del 26 de Junio y el resultado del partido entre Argentina y Honduras.
3. Verifica en las páginas oficiales: Puedes buscar en las páginas oficiales de la Asociación del Fútbol Argentino (AFA) o la Federación Nacional Autónoma de Fútbol de Honduras (FENAFUTH) para ver si hay un comunicado o un reporte sobre el partido.

Sin embargo, te puedo decir que Arg

# EJEMPLO CON CONTEXTO ACTUALIZADO POR WEB SCRAPPING

In [3]:

# Se van a ignorar Warnings  asociados a las versiones en uso
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=ResourceWarning)

# El scrapping se ejecuta en 3 etapas
# 1) Realizamos la busqueda mediante ddgs
# 2) Descargamos los datos en memoria mediante requests
# 3) Formateamos el HTML con BeautifulSoup para extraer los datos especificos
import requests
from bs4 import BeautifulSoup
from ddgs import DDGS



def buscador(query:str)->str:
  """
  Función para realizar una busqueda en internet y extraer el resultado.
  in:
    query: str
  out:
    url: str
    title: str
    text: str
  """

  # Definimos el header que va a recibir como meta dato el motor de busqueda
  headers = {"User-Agent": "Mozilla/5.0"}

  # Corremos el metodo de duckduckgo para la busqueda en internet
  with DDGS() as ddgs:
      results = list(ddgs.text(query, max_results=1))

  # Extraemos la url de la busqueda
  r = results[0]
  url = r["href"]

  # Con la URL resultado de la busqueda, descargamos el html con request y lo formateamos con BeautifulSoup
  response = requests.get(url, headers=headers)
  soup = BeautifulSoup(response.text, "html.parser")

  for tag in soup(["script", "style"]):
      tag.decompose()

  # Extraemos el titulo y el texto
  title = soup.title.string if soup.title else ""
  text = " ".join(soup.get_text().split())

  return (url, title, text)


In [8]:
# Preparamos una consulta
query = "Resultado final partido Argentina vs Honduras 06/06/2026"

# Llamamos a la funcion y obtenemos los resultados
url, title, text = buscador(query)

# Formateamos el contexto pasando como argumento los datos obtenidos
contexto = f"""
FUENTE:
Título: {title}
URL: {url}
Contenido: {text[:1000]}
"""


# Realizamos la misma pregunta que probamos anteriormente
pregunta = "Quien fue el ganador, y por cuanto gano, del partido de futbol de Argentina contra Honduras en Junio del 26"

# Ejecutamos la llamada al LLM
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": f"""
Usa el siguiente contexto para responder:

{contexto}

PREGUNTA:
{pregunta}
"""
        }
    ]
)

print(response.choices[0].message.content)


Según la información proporcionada, el ganador del partido de fútbol entre Argentina y Honduras fue Argentina. Argentina ganó con un marcador de 2 a 0. 

El gol de Argentina fue anotado por dos jugadores: Lautaro Martínez en el minuto 37' (penal) y Giuliano Simeone en el minuto 54'.


En este ejemplo podemos ver como un mismo modelo, misma pregunta, tenemos dos resultados completamente distintos, esto es debido a que le agregamos el contexto necesario al modelo para que pueda responder a la pregunta de forma certera.

- Cabe aclarar que al momento de realizar esta practica la información no estaba indexada por el modelo por lo cual el resultado fue distinto, si al leer este manual el resultado no difiere sustancialmente se recomienda hacer pruebas con hechos o sucesos actuales al momento de la ejecucion
